# MAMA-MIA preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.mama_mia import MamaMia, csv_save_path
from utils.preprocessing import UNKNOWN, breast_density, get_value, isna_v2, race_mappings, yes_no_mapping

ds = MamaMia()
ds.set_dataset_name("mama-mia")
ds.clinical_df.shape, ds.clinical_df.head()

### Step 1 — drop columns not used by the common schema (treatment, staging, acquisition metadata, etc.)

In [ ]:
ds.clinical_df = ds.clinical_df.drop(columns=[
    "nac agent", "endocrine therapy", "anti her2 neu therapy", "pcr", "mastectomy post nac",
    "days to follow up", "days to recurrence", "days to metastasis", "days to death",
    "hr", "er", "pr", "her2", "mammaprint", "oncotype score", "nottingham grade", "tumor subtype",
    "patient size", "weight", "high bit", "window center", "window width", "field strength",
    "fat suppressed", "image rows", "image columns", "num slices", "pixel spacing", "slice thickness",
    "site", "echo time", "repetition time", "acquisition date", "tcia series uid",
])
ds.clinical_df.shape

### Step 2 — map the remaining clinical fields to harmonized labels

In [ ]:
ds.clinical_df["bilateral breast cancer"] = ds.clinical_df["bilateral breast cancer"].apply(lambda x: get_value(x, yes_no_mapping))
ds.clinical_df["multifocal cancer"] = ds.clinical_df["multifocal cancer"].apply(
    lambda x: get_value(int(x), yes_no_mapping) if not isna_v2(x) else None
)
ds.clinical_df["ethnicity"] = ds.clinical_df["ethnicity"].apply(lambda x: get_value(x, race_mappings) if not isna_v2(x) else UNKNOWN)
ds.clinical_df["has implant"] = ds.clinical_df["has implant"].apply(lambda x: get_value(x, yes_no_mapping))
ds.clinical_df["breast density"] = ds.clinical_df["breast density"].apply(lambda x: get_value(x, breast_density) if not isna_v2(x) else None)
ds.clinical_df["bilateral mri"] = ds.clinical_df["bilateral mri"].apply(lambda x: get_value(x, yes_no_mapping))
ds.clinical_df["bmi group"] = ds.clinical_df["bmi group"].apply(lambda x: " ".join(x.split("_")) if not isna_v2(x) else None)
ds.clinical_df[["ethnicity", "breast density", "bilateral mri"]].apply(lambda c: c.value_counts(dropna=False))

## Build exam records for one patient

In [ ]:
patient_id, patient_df = next(iter(ds.clinical_df.groupby("patient id")))
exams = ds.process_patient(patient_df)
len(exams), exams[0] if exams else None

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"clinical rows (1 per patient): {len(ds.clinical_df)} | rows in final csv (1+ slices per patient): {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))